# Bước 4b: giải thích quyết định của CNN và kiểm chứng độ trung thực (Colab)

Notebook huấn luyện lại 5 mô hình `cnn_AT_calonly` của Bước 3 (seed 0–4, cùng code và seed; Bước 3 không lưu trọng số) và kiểm tra dự đoán có trùng với Bước 3 không. Sau đó tính lời giải thích Integrated Gradients và gradient × đầu vào cho mùa vụ 4 và 5, chạy kiểm tra xóa (dải xếp hạng cao nhất so với dải ngẫu nhiên, 100 lần rút) và kiểm tra độ ổn định. Thiết kế chi tiết ở đầu file `src/step4b_explain_cnn.py`.

**Chuẩn bị:**
1. File `MangoDMC_NIR_Data_v3.csv` đã có trong `MyDrive/Mango_NIR/data/` (như Bước 3).
2. Chọn *Runtime → Change runtime type → T4 GPU*. Nếu phiên không có GPU, script không huấn luyện mô hình (Bước 3 huấn luyện trên GPU); chỉ bước chạy thử được chạy trên CPU.
3. Chạy lần lượt từng ô. Nếu bước **chạy thử** (mục 5) báo lỗi, dừng lại và gửi thông báo lỗi cho Claude.

**Thời gian:** huấn luyện 5 mô hình mất khoảng 15–20 phút (Bước 3: 2,3–4,3 phút mỗi mô hình), phần giải thích và kiểm tra vài phút. Mô hình được lưu trong `MyDrive/Mango_NIR/step4b_results/models/`. Script không bắt đầu seed mới khi đã chạy quá 4 giờ. Nếu script dừng vì hết giờ hoặc Colab ngắt kết nối, chạy lại notebook từ đầu: mô hình đã lưu được dùng lại, không huấn luyện lại.

**Kết quả:** `MyDrive/Mango_NIR/step4b_results/`; ô cuối tạo file nén `MyDrive/Mango_NIR/step4b_results.zip`.

## 1. Kiểm tra GPU và TensorFlow

In [ ]:
%env TF_CPP_MIN_LOG_LEVEL=2
!nvidia-smi -L
try:
    import tensorflow as tf
except ImportError:  # in case the Colab image no longer ships TensorFlow
    !pip install -q tensorflow
    import tensorflow as tf
GPUS = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| GPU:', GPUS)
if not GPUS:
    print('CẢNH BÁO: phiên này không có GPU, nên mục 6 sẽ không huấn luyện mô hình. '
          'Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại từ đầu.')

## 2. Kết nối Google Drive

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/Mango_NIR')
DATA_CSV = DRIVE / 'data' / 'MangoDMC_NIR_Data_v3.csv'
OUT = DRIVE / 'step4b_results'
assert DATA_CSV.exists(), f'Chưa thấy file dữ liệu {DATA_CSV}. Upload file vào đó rồi chạy lại ô này.'
OUT.mkdir(parents=True, exist_ok=True)
print('Dữ liệu:', DATA_CSV, '| Kết quả:', OUT)

## 3. Lấy mã nguồn từ GitHub

In [ ]:
import sys

REPO_URL = 'https://github.com/dohieunghia2002/mango_nir.git'
REPO = Path('/content/mango_nir')
if not REPO.exists():
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log --oneline -1
link = REPO / 'data' / DATA_CSV.name
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DATA_CSV)
%cd {REPO}
sys.path.insert(0, str(REPO))

import subprocess

from src.step4b_explain_cnn import EXIT_NO_GPU, EXIT_STEP3_MISMATCH, EXIT_TIME_BUDGET

EXIT_HINTS = {  # what to do after each exit code of src.step4b_explain_cnn
    EXIT_TIME_BUDGET: 'đã chạy quá 4 giờ. Chạy lại notebook từ đầu trong phiên mới: mô hình đã lưu được dùng lại.',
    EXIT_STEP3_MISMATCH: 'dự đoán của mô hình khác Bước 3. Gửi thông báo ở trên cho Claude; mô hình đã lưu vẫn được giữ.',
    EXIT_NO_GPU: 'phiên này không có GPU nên script không huấn luyện mô hình. '
                 'Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại từ đầu.',
}


def run_module(*args):
    """Run `python -m <args>` in the repo, print its output as it comes, and stop the notebook if it fails."""
    p = subprocess.Popen([sys.executable, '-u', '-m', *args], cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        hint = EXIT_HINTS.get(p.returncode, 'xem thông báo ở trên.')
        raise RuntimeError(f'{args[0]} dừng với mã {p.returncode}: {hint} Các ô sau chưa được chạy.')

## 4. Kiểm tra dữ liệu (dừng với lỗi nếu số phổ hoặc số quả không khớp)

In [ ]:
from src.data import count_table, load_data

print(count_table(load_data()))

## 5. Chạy thử

1 seed, 2 epoch trên 1.000 phổ, 40 quả mỗi tập kiểm thử, ít lần rút; chỉ để kiểm tra mã nguồn. Kết quả để ở `/content/smoke_step4b`, không ghi lên Drive. Bước này chạy được cả trên CPU. Dòng so sánh với Bước 3 sẽ ghi `DIFFERENT` vì mô hình chỉ học 2 epoch; điều này là bình thường.

In [ ]:
import shutil

shutil.rmtree('/content/smoke_step4b', ignore_errors=True)  # fresh folder: the smoke model is trained again
run_module('src.step4b_explain_cnn', '--smoke', '--out', '/content/smoke_step4b')

## 6. Chạy đầy đủ (5 seed)

Với mỗi seed, script in hai dòng:
- mô hình được huấn luyện mới (`trained`) hay dùng lại (`loaded`); sai lệch lớn nhất so với dự đoán của Bước 3 trên cả 12.011 phổ; RMSEP mới / RMSEP của Bước 3; môi trường đã huấn luyện mô hình;
- nửa độ rộng khoảng, sai số tính đầy đủ của Integrated Gradients (trung vị / lớn nhất, %FW) và thời gian chạy.

Có thể chạy lại ô này sau khi bị ngắt kết nối. Script tự dừng (ô báo lỗi) trong ba trường hợp:
- **Phiên không có GPU** mà vẫn còn mô hình chưa huấn luyện: đổi sang T4 GPU rồi chạy lại từ đầu.
- **Dự đoán khác Bước 3** quá 1e-4 %FW: mô hình đã lưu được giữ lại, seed đó chưa được phân tích. Gửi thông báo cho Claude để quyết định. Chỉ khi đã thống nhất chạy tiếp mới đặt `ACCEPT_STEP3_MISMATCH = True` trong ô dưới rồi chạy lại ô này; lựa chọn này được ghi vào kết quả.
- **Đã chạy quá 4 giờ** (tính từ lúc ô này bắt đầu) khi sắp bắt đầu một seed mới: chạy lại notebook từ đầu trong phiên mới.

In [ ]:
ACCEPT_STEP3_MISMATCH = False  # True only after agreeing with Claude (see the text above)
run_module('src.step4b_explain_cnn', '--out', str(OUT), *(['--accept-step3-mismatch'] if ACCEPT_STEP3_MISMATCH else []))

## 7. Đóng gói kết quả

In [ ]:
import json
import zipfile

FILES = ['step4b.json', 'step4b_faithfulness.csv', 'step4b_stability.csv', 'step4b_explanations.csv']
missing = [f for f in FILES if not (OUT / f).exists()]
if missing:
    raise RuntimeError(f'Chưa có kết quả {missing}: chạy mục 6 trước khi đóng gói.')
if json.loads((OUT / 'step4b.json').read_text(encoding='utf-8'))['config']['smoke']:
    raise RuntimeError('Kết quả trong thư mục là của lần chạy thử, không đóng gói.')
ZIP = DRIVE / 'step4b_results.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        z.write(OUT / f, Path('results') / f)
print('Đã tạo', ZIP, f'({ZIP.stat().st_size / 1e6:.1f} MB)')

**Bước tiếp theo:** tải `step4b_results.zip` từ Drive về máy, giải nén vào thư mục gốc của repo (tạo `results/step4b*`), rồi báo Claude để phân tích và ghi vào `results/summary.json`.